In [ ]:
!pip install git+https://github.com/openai/CLIP.git

!git clone https://github.com/victoriachernova/styleclip-nada.git
%cd styleclip-nada

!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git /content/stylegan2-ada-pytorch

!mkdir -p weights
!wget -q -O weights/ffhq.pkl https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl

In [ ]:
import sys
import torch
from huggingface_hub import hf_hub_download
import matplotlib.pyplot as plt
import torchvision.utils as vutils

sys.path.insert(0, '..')

from src.models.generator import Generator
from src.models.mapper import Mapper
from src.utils import show_grid

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

G = Generator('weights/ffhq.pkl', device=device)

In [ ]:
styles = {
    'anime': 'mapper_an_anime_portrait.pt',
    'zombie': 'mapper_a_photo_of_a_person_as_a_zombie.pt',
    'vampire': 'mapper_a_realistic_vampire.pt'
}
mappers = {}

for style, filename in styles.items():

  mapper = Mapper(latent_dim=512, hidden_dim=1024, num_layers=4).to(device)

  ckpt = hf_hub_download(
    repo_id = 'tvictoria/styleclip-nada-ffhq',
    filename=filename
)

  checkpoint = torch.load(ckpt, map_location=device)

  mapper.load_state_dict(checkpoint['state_dict'])

  mapper.eval()

  mappers[style] = mapper

In [ ]:
seed = 42

torch.manual_seed(seed)

z = torch.randn(4, G.G.z_dim, device=device)

with torch.no_grad():

  w = G.mapping(z)

  img_before = G.synthesis(w)

In [ ]:
results = {}

with torch.no_grad():

  for style, mapper in mappers.items():

    delta = mapper(w)

    img = G.synthesis(w + delta)

    results[style] = img.cpu()

In [ ]:
before = F.interpolate(img_before.cpu(), size=256)
anime = F.interpolate(results['anime'].cpu(), size=256)
zombie = F.interpolate(results['zombie'].cpu(), size=256)
vampire = F.interpolate(results['vampire'].cpu(), size=256)

fig, axes = plt.subplots(1, 4, figsize=(20,5))

images = [before, anime, zombie, vampire]
titles = ['Before', 'Anime', 'Zombie', 'Vampire']

for ax, imgs, title in zip(axes, images, titles):

    grid = vutils.make_grid((imgs.clamp(-1,1)+1)/2, nrow=2)

    ax.imshow(grid.permute(1,2,0))
    ax.set_title(title, fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()